In [2]:
from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder

import numpy as np
import chromadb
import anthropic
from dotenv import load_dotenv


In [67]:
import os
key = os.environ.get("ANTHROPIC_API_KEY", "")
print("有讀到金鑰" if key.startswith("sk-ant-") else "沒讀到金鑰，檢查 .env")

有讀到金鑰


In [41]:
def load(folder: str) -> list[dict]:
    docs = []
    for p in Path(folder).glob("*.md"):
        text = p.read_text(encoding="utf-8")
        docs.append({"text": text, "source": p.name})
    return docs

In [42]:
docs = load("data")
lines = docs[0]["text"].splitlines()

In [43]:
def split(docs: list[dict]) -> list[dict]:
    chunks = []
    for doc in docs:
        section = ""
        for line in doc["text"].splitlines():
            line = line.strip()
            if line.startswith("## "):
                section = line[3:]
            elif line and not line.startswith("#") and not line.startswith(">"):
                chunks.append({
                    "text": f"【{section}】{line}",
                    "source": doc["source"],
                    "section": section,
                })
    return chunks

In [44]:
chunks = split(docs)
print(f"共 {len(chunks)} 塊")
for c in chunks[:4]:
    print(c)

共 57 塊
{'text': '【店家基本資訊】茶語時光是一家位於台北市大安區的手搖飲料店，地址為台北市大安區和平東路二段 88 號。', 'source': 'tea_shop_kb.md', 'section': '店家基本資訊'}
{'text': '【店家基本資訊】門市電話為 02-2700-1234，官方 LINE 帳號為 @teatime88。', 'source': 'tea_shop_kb.md', 'section': '店家基本資訊'}
{'text': '【營業時間】週一至週五營業時間為上午 10:00 至晚上 9:30。', 'source': 'tea_shop_kb.md', 'section': '營業時間'}
{'text': '【營業時間】週六、週日及國定假日營業時間為上午 11:00 至晚上 10:00。', 'source': 'tea_shop_kb.md', 'section': '營業時間'}


In [45]:
model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

In [55]:
def embed(texts: list[str]) -> np.ndarray:
    return model.encode(texts,
                        normalize_embeddings = True,
                        show_progress_bar=False)

In [56]:
texts = [c["text"] for c in chunks]
vectors = embed(texts)
print(len(texts), vectors.shape)

57 (57, 512)


In [57]:
client = chromadb.PersistentClient(path= "./db")
# client = chromadb.EphemeralClient()
# client.delete_collection("tea_shop")
col = client.get_or_create_collection("tea_shop")
print(col.count())

0


In [58]:
def store(chunks: list[dict], vectors: np.ndarray) -> None:
    col.upsert(
        ids=[f"chunk-{i}" for i in range(len(chunks))],
        documents=[c["text"] for c in chunks],
        embeddings=vectors.tolist(),
        metadatas=[{"source": c["source"], "section": c["section"]} for c in chunks],
    )

In [75]:
def build_index(folder: str = ".") -> None:
    """重建整個索引：刪掉舊的 collection，重新 load → split → embed → store"""
    global col
    try:
        chromadb.delete_collection("tea_shop")
    except Exception:
        pass
    col = chromadb.get_or_create_collection("tea_shop")

    chunks = split(load(folder))
    store(chunks, embed([c["text"] for c in chunks]))
    print(f"索引完成，共 {col.count()} 塊")

In [76]:
store(chunks, vectors)
print(col.count())
# print(col.get(ids=["chunk-42"], include=["documents", "metadatas"]))

57


In [77]:
q = embed(["孕婦可以喝什麼？"])
res = col.query(query_embeddings=q.tolist(), n_results=3)
print(res["documents"])
print(res["metadatas"])
print(res["distances"])

[['【咖啡因說明】孕婦或對咖啡因敏感的顧客，建議選擇蕎麥茶或鮮果飲。', '【加料選項】每杯飲料最多可加三種配料。', '【奶茶與拿鐵系列】鮮奶茶：中杯 60 元，大杯 70 元。使用 100% 林鳳營鮮乳，不含奶精。']]
[[{'section': '咖啡因說明', 'source': 'tea_shop_kb.md'}, {'section': '加料選項', 'source': 'tea_shop_kb.md'}, {'source': 'tea_shop_kb.md', 'section': '奶茶與拿鐵系列'}]]
[[0.7703853249549866, 1.0296642780303955, 1.0343759059906006]]


In [78]:
d = res["distances"][0][0]
print(f"距離 {d:.3f} → 相似度 {1 - d/2:.3f}")

距離 0.770 → 相似度 0.615


In [79]:
def retrieve(question: str, top_k: int = 5) -> list[dict]:
    q = embed([question])
    res = col.query(query_embeddings=q.tolist(), n_results=top_k)
    results = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        results.append({
            "text": doc,
            "source": meta["source"],
            "section": meta["section"],
            "score": 1 - dist / 2,
        })
    return results

In [80]:
for q in ["孕婦可以喝什麼？", "刷卡有最低消費嗎？", "芒果冰沙可以做熱的嗎？"]:
    print(f"\n問題：{q}")
    for r in retrieve(q, top_k=5):
        print(f"  {r['score']:.3f}  {r['text']}")


問題：孕婦可以喝什麼？
  0.615  【咖啡因說明】孕婦或對咖啡因敏感的顧客，建議選擇蕎麥茶或鮮果飲。
  0.485  【加料選項】每杯飲料最多可加三種配料。
  0.483  【奶茶與拿鐵系列】鮮奶茶：中杯 60 元，大杯 70 元。使用 100% 林鳳營鮮乳，不含奶精。
  0.482  【咖啡因說明】可可鮮奶含有微量咖啡因，大杯約 10 毫克，對咖啡因敏感者請斟酌飲用。
  0.472  【奶茶與拿鐵系列】可可鮮奶：中杯 65 元，大杯 75 元。比利時可可粉加鮮奶，不含茶。

問題：刷卡有最低消費嗎？
  0.606  【付款方式】信用卡僅接受單筆消費 100 元以上。
  0.539  【會員制度】每消費 50 元累積 1 點，集滿 10 點可兌換任一中杯飲品一杯。
  0.507  【付款方式】門市接受現金、LINE Pay、街口支付、悠遊卡與信用卡。
  0.492  【營業時間】每月最後一個週二為店休日，當天不營業。
  0.459  【會員制度】點數有效期限為累積後 12 個月，逾期自動失效。

問題：芒果冰沙可以做熱的嗎？
  0.709  【甜度與冰塊調整】冰沙類飲品無法做熱飲或溫飲。
  0.665  【甜度與冰塊調整】芒果冰沙與柳橙鮮果飲無法調整為無糖，最低為微糖。
  0.631  【水果茶系列】芒果冰沙：大杯 75 元，僅提供大杯。季節限定，每年 5 月至 9 月販售。
  0.564  【甜度與冰塊調整】冰塊可選擇：正常冰、少冰、微冰、去冰、溫、熱。
  0.476  【咖啡因說明】以下飲品不含咖啡因：蕎麥茶、黑糖珍珠鮮奶、柳橙鮮果飲、芒果冰沙。


In [73]:
# reranker = CrossEncoder("BAAI/bge-reranker-base")
#
# def rerank(question: str, chunks: list[dict], top_k: int = 5) -> list[dict]:
#     scores = reranker.predict([(question, c["text"]) for c in chunks])
#     ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
#     return [c for c, s in ranked[:top_k]]

一個好的 RAG 提示詞有 4 個部分

① 角色：你是誰、在做什麼
② 規則：只能根據資料回答、找不到怎麼辦、要不要附出處
③ 資料：retrieve() 找到的內容，加上編號
④ 問題：使用者的提問

In [81]:
def build_prompt(question: str, chunks: list[dict]) -> str:
    """把問題和檢索到的資料組成提示詞"""
    lines = [f"[{i}] {c['text']}" for i, c in enumerate(chunks, start=1)]
    context = "\n".join(lines)

    return f"""你是「茶語時光」手搖飲店的客服助理。

規則：
1. 只能根據 <資料> 中的內容回答，不要自己推測或補充。
2. 如果資料中沒有答案，請回答「抱歉，這個問題我無法回答，請洽門市 02-2700-1234。」
3. 回答要簡短、親切，使用繁體中文。
4. 在回答最後標出參考的資料編號，例如：（參考：[1][3]）

<資料>
{context}
</資料>

<問題>
{question}
</問題>"""

In [82]:
q = "芒果冰沙可以做熱的嗎？"
print(build_prompt(q, retrieve(q)))

你是「茶語時光」手搖飲店的客服助理。

規則：
1. 只能根據 <資料> 中的內容回答，不要自己推測或補充。
2. 如果資料中沒有答案，請回答「抱歉，這個問題我無法回答，請洽門市 02-2700-1234。」
3. 回答要簡短、親切，使用繁體中文。
4. 在回答最後標出參考的資料編號，例如：（參考：[1][3]）

<資料>
[1] 【甜度與冰塊調整】冰沙類飲品無法做熱飲或溫飲。
[2] 【甜度與冰塊調整】芒果冰沙與柳橙鮮果飲無法調整為無糖，最低為微糖。
[3] 【水果茶系列】芒果冰沙：大杯 75 元，僅提供大杯。季節限定，每年 5 月至 9 月販售。
[4] 【甜度與冰塊調整】冰塊可選擇：正常冰、少冰、微冰、去冰、溫、熱。
[5] 【咖啡因說明】以下飲品不含咖啡因：蕎麥茶、黑糖珍珠鮮奶、柳橙鮮果飲、芒果冰沙。
</資料>

<問題>
芒果冰沙可以做熱的嗎？
</問題>


In [83]:
load_dotenv()
llm = anthropic.Anthropic()


In [84]:
def generate(prompt: str) -> str:
    msg = llm.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in msg.content if b.type == "text")

In [85]:
def ask(question: str, top_k: int = 3) -> str:
    chunks = retrieve(question, top_k)
    prompt = build_prompt(question, chunks)
    return generate(prompt)

In [86]:
questions = [
    "芒果冰沙可以做熱的嗎？",
    "刷卡有最低消費嗎？",
    "孕婦可以喝什麼？",
    "你們有賣咖啡嗎？",
]
for q in questions:
    print(f"🙋 {q}")
    print(f"🤖 {ask(q)}\n")

🙋 芒果冰沙可以做熱的嗎？
🤖 不好意思，芒果冰沙無法做熱飲或溫飲喔。（參考：[1]）

🙋 刷卡有最低消費嗎？
🤖 有的，信用卡僅接受單筆消費 100 元以上喔！未滿 100 元的話，也可以改用現金、LINE Pay、街口支付或悠遊卡付款。😊（參考：[1][3]）

🙋 孕婦可以喝什麼？
🤖 孕婦或對咖啡因敏感的顧客，建議選擇蕎麥茶或鮮果飲喔！😊（參考：[1]）

🙋 你們有賣咖啡嗎？
🤖 抱歉，咖啡系列目前暫無販售喔！若想嘗試其他飲品，歡迎參考我們的茶類與鮮果飲系列。😊（參考：[1]）

